# Prepare a lunar elevation grid
Run with a Python CPU runtime. This notebook exports a little-endian float32 grid and JSON metadata. The default LOLA global grid runs north to south and west to east. When switching datasets, update the configuration overrides and confirm the vertical units and datum before exporting.

In [ ]:
%pip -q install rasterio matplotlib requests

In [ ]:
from pathlib import Path
from urllib.parse import urlparse
import json

import matplotlib.pyplot as plt
import numpy as np
import rasterio
import requests
from rasterio.enums import Resampling

# Defaults describe the global LOLA JP2. For another raster, clear the
# overrides to use its embedded metadata or supply its documented values.
SOURCE = "https://imbrium.mit.edu/DATA/LOLA_GDR/CYLINDRICAL/JP2/LDEM_16.JP2"
PRODUCT_LABEL = "https://imbrium.mit.edu/DATA/LOLA_GDR/CYLINDRICAL/JP2/LDEM_16_JP2.LBL"
MAX_WIDTH = 1024
SCALE_OVERRIDE = 0.5
OFFSET_OVERRIDE = 0.0  # Relative elevation in meters, not absolute radius.
REFERENCE_RADIUS_M = 1737400
GLOBAL_EXTENT_DEG = [-180, -90, 180, 90]  # Set to None for a regional raster.
ASSET_NAME = "global-height"
WORK = Path("data/work")
EXPORT = WORK / "exports"
WORK.mkdir(parents=True, exist_ok=True)
EXPORT.mkdir(exist_ok=True)

url = urlparse(SOURCE)
if url.scheme in ("https", "http"):
    raster_path = WORK / Path(url.path).name
    if not raster_path.exists():
        temporary_path = raster_path.with_suffix(raster_path.suffix + ".part")
        with requests.get(SOURCE, stream=True, timeout=(15, 120)) as response:
            response.raise_for_status()
            with temporary_path.open("wb") as output:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    output.write(chunk)
        temporary_path.replace(raster_path)
else:
    raster_path = Path(SOURCE)

In [ ]:
with rasterio.open(raster_path) as dataset:
    width = min(MAX_WIDTH, dataset.width)
    height = max(1, round(dataset.height * width / dataset.width))
    if GLOBAL_EXTENT_DEG is not None and dataset.width != dataset.height * 2:
        raise ValueError("The global extent override expects a 2:1 raster.")
    if dataset.crs is None and GLOBAL_EXTENT_DEG is None:
        raise ValueError("The raster needs georeferencing before it can be exported.")
    scale = dataset.scales[0] if SCALE_OVERRIDE is None else SCALE_OVERRIDE
    offset = dataset.offsets[0] if OFFSET_OVERRIDE is None else OFFSET_OVERRIDE
    values = dataset.read(1, out_shape=(height, width), masked=True,
                          out_dtype="float32", resampling=Resampling.average)
    elevation = np.asarray((values * scale + offset).filled(np.nan), dtype="<f4")
    transform = dataset.transform * dataset.transform.scale(
        dataset.width / width, dataset.height / height
    )
    metadata = {
        "source": SOURCE, "productLabel": PRODUCT_LABEL,
        "width": width, "height": height, "encoding": "float32-le",
        "units": "m", "order": "row-major", "nodataEncoding": "NaN",
        "referenceRadiusM": REFERENCE_RADIUS_M,
        "globalExtentDegrees": GLOBAL_EXTENT_DEG,
        "crsWkt": dataset.crs.to_wkt() if dataset.crs else None,
        "affine": list(transform)[:6] if dataset.crs else None,
        "sourceScale": scale, "sourceOffset": offset, "resampling": "average",
    }

if not np.isfinite(elevation).any():
    raise ValueError("No valid elevations found.")
metadata["minElevationM"] = float(np.nanmin(elevation))
metadata["maxElevationM"] = float(np.nanmax(elevation))
elevation.tofile(EXPORT / f"{ASSET_NAME}.f32")
(EXPORT / f"{ASSET_NAME}.json").write_text(json.dumps(metadata, indent=2) + "\n")
print(json.dumps(metadata, indent=2))
plt.imshow(elevation, cmap="terrain")
plt.colorbar(label="Elevation (m)")
plt.show()

from zipfile import ZipFile, ZIP_DEFLATED
with ZipFile(WORK / f"{ASSET_NAME}.zip", "w", compression=ZIP_DEFLATED) as bundle:
    for suffix in ("f32", "json"):
        path = EXPORT / f"{ASSET_NAME}.{suffix}"
        bundle.write(path, path.name)
print(f"Download {WORK / (ASSET_NAME + '.zip')} from the notebook file panel.")